In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from losses.final_loss import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    final_PatchDataset_cbam,
    evaluate_full_volume_cbam,
    compute_per_class_dice
    
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.7
NUM_WORKERS = 2
train_dataset = final_PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = final_PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [6]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [7]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)
# weights = torch.tensor([0.05, 1, 1.2, 1.5, 2.1, 2.1, 1])
weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.6,
    class_weights = weights
)
scaler = torch.cuda.amp.GradScaler()


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_27220\1922899496.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [21]:
def train_one_epoch_cbam(model, loader, optimizer, scaler, loss_fn, device):

    model.train()
    total_loss = 0

    for images, labels, dist_maps in tqdm(loader):

        images = images.to(device)
        labels = labels.long().to(device)
        dist_maps = dist_maps.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            # Deep supervision unpack
            out, ds2, ds3, ds4 = outputs


            loss_main = loss_fn(out, labels, dist_maps)
            loss_ds2  = loss_fn(ds2, labels, dist_maps)
            loss_ds3  = loss_fn(ds3, labels, dist_maps)
            loss_ds4  = loss_fn(ds4, labels, dist_maps)

            loss = (
                1.0 * loss_main +
                0.5 * loss_ds2 +
                0.25 * loss_ds3 +
                0.125 * loss_ds4
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch_cbam(model, loader, loss_fn, device, num_classes=7):

    model.eval()
    total_loss = 0
    all_dices = []

    with torch.no_grad():
        for images, labels, dist_maps in loader:

            images = images.to(device)
            labels = labels.long().to(device)
            dist_maps = dist_maps.to(device)

            with torch.amp.autocast("cuda"):

                outputs = model(images)
                out = outputs[0]

                loss = loss_fn(out, labels, dist_maps)

            total_loss += loss.item()

            batch_dice = compute_per_class_dice(out, labels, num_classes)
            all_dices.append(batch_dice)

    mean_loss = total_loss / len(loader)

    all_dices = np.array(all_dices)
    mean_dices = np.mean(all_dices, axis=0)

    return mean_loss, mean_dices

### weights ([0.05, 1, 1, 1.2, 1.8, 1.8, 1])

In [22]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "06_iw_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "06_iw_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "06_iw_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "06_iw_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "06_iw_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 198/198 [10:32<00:00,  3.20s/it]



Epoch 0
Train Loss: 0.2847
Val Loss: 0.0864
Per Class Dice: [0.82511721 0.84986071 0.83143195 0.80892929 0.83400037 0.78543217]
Mean Dice: 0.8219
Parotid Dice (L,R): 0.8089, 0.8340
Combined Score: 0.8218
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [10:03<00:00,  3.05s/it]



Epoch 1
Train Loss: 0.2798
Val Loss: 0.1042
Per Class Dice: [0.97214821 0.77984339 0.95970546 0.98245734 0.82129707 0.73806993]
Mean Dice: 0.8563
Parotid Dice (L,R): 0.9825, 0.8213
Combined Score: 0.8700
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000199


100%|██████████| 198/198 [13:46<00:00,  4.18s/it]


...


Epoch 11
Train Loss: 0.2761
Val Loss: 0.0737
Per Class Dice: [0.58576183 0.78239449 0.8709152  0.37307379 0.58138197 0.69992449]
Mean Dice: 0.6615
Parotid Dice (L,R): 0.3731, 0.5814
Combined Score: 0.6062
Saved Best Loss Model
LR: 0.000084


100%|██████████| 198/198 [12:45<00:00,  3.86s/it]



Epoch 12
Train Loss: 0.2484
Val Loss: 0.0838
Per Class Dice: [0.9430647  0.8757062  0.8572064  0.86037635 0.7853157  0.90130003]
Mean Dice: 0.8560
Parotid Dice (L,R): 0.8604, 0.7853
Combined Score: 0.8460
LR: 0.000069


  7%|▋         | 14/198 [00:41<09:08,  2.98s/it]


KeyboardInterrupt: 

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM().to(device)

model_path = "../experiments/custom_loss_cbam/06_iw_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_25576\2716405960.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)

Model loaded for testing.


In [10]:
import nibabel as nib
def get_gaussian_weight_map(patch_size):
    import numpy as np

    ranges = [np.linspace(-1, 1, s) for s in patch_size]
    z, y, x = np.meshgrid(*ranges, indexing="ij")

    dist = z**2 + y**2 + x**2
    gaussian = np.exp(-dist / 0.5)

    gaussian = gaussian / np.max(gaussian)
    return torch.tensor(gaussian, dtype=torch.float32)


def final_sliding_window_cbam(model, image, patch_size=96, stride=48, device="cuda"):
    model.eval()

    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    _, _, D, H, W = image_t.shape
    num_classes = 7

    # CPU accumulation (safe for memory)
    output     = torch.zeros((1, num_classes, D, H, W), dtype=torch.float32)
    weight_map = torch.zeros_like(output)

    gaussian = get_gaussian_weight_map((patch_size, patch_size, patch_size))
    gaussian_gpu = gaussian.unsqueeze(0).unsqueeze(0).to(device)
    gaussian_cpu = gaussian.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        z_steps = sorted(set(list(range(0, max(1, D - patch_size), stride)) + [max(0, D - patch_size)]))
        y_steps = sorted(set(list(range(0, max(1, H - patch_size), stride)) + [max(0, H - patch_size)]))
        x_steps = sorted(set(list(range(0, max(1, W - patch_size), stride)) + [max(0, W - patch_size)]))

        print(f"Volume: {D}x{H}x{W} | Patches: {len(z_steps)*len(y_steps)*len(x_steps)} | Stride: {stride}")

        for z in z_steps:
            for y in y_steps:
                for x in x_steps:

                    patch = image_t[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size]

                    # ---- Forward ----
                    logits = model(patch)[0]   # deep supervision → take main output
                    probs = torch.softmax(logits, dim=1)

                    # ---- Weight with Gaussian ----
                    probs = probs * gaussian_gpu

                    # ---- Move to CPU once ----
                    probs_cpu = probs.cpu()

                    output[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += probs_cpu
                    weight_map[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += gaussian_cpu

    # ---- Normalize ----
    output = output / weight_map.clamp(min=1e-6)

    # ---- Argmax ----
    pred = torch.argmax(output, dim=1).squeeze().numpy()

    return pred


def final_evaluate_full_volume_cbam(model, cases, images_dir, labels_dir,
                             patch_size=96, stride=48, device="cuda"):

    model.eval()
    all_dices = []

    for case in cases:

        image = nib.load(f"{images_dir}/{case}.nii.gz").get_fdata(dtype=np.float32)
        label = nib.load(f"{labels_dir}/{case}.nii.gz").get_fdata().astype(np.uint8)

        pred = final_sliding_window_cbam(
            model,
            image,
            patch_size=patch_size,
            stride=stride,
            device=device
        )

        print(f"\nCase: {case}")
        print("Pred labels:", np.unique(pred))
        print("GT labels:", np.unique(label))

        case_dices = []

        for cls in range(1, 7):  # ignore background

            pred_cls = (pred == cls)
            label_cls = (label == cls)

            intersection = np.sum(pred_cls & label_cls)
            union = np.sum(pred_cls) + np.sum(label_cls)

            # ---- SAFE Dice ----
            if union == 0:
                dice = 1.0  # both empty → perfect
            else:
                dice = (2 * intersection) / union

            case_dices.append(dice)

        all_dices.append(case_dices)

        print(f"Dice: {np.round(case_dices, 4)}")

    all_dices = np.array(all_dices)

    mean_dice = np.mean(all_dices, axis=0)
    std_dice  = np.std(all_dices, axis=0)

    print("\n===== FINAL RESULTS =====")
    print("Mean Dice:", np.round(mean_dice, 4))
    print("Std Dice :", np.round(std_dice, 4))

    return mean_dice, std_dice



In [12]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9015 0.8635 0.8061 0.8195 0.8295 0.777 ]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9135 0.8058 0.7084 0.8163 0.8567 0.8798]
Volume: 264x333x333 | Patches: 180 | Stride: 48

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.915  0.8208 0.8174 0.7818 0.8072 0.8648]
Volume: 248x395x395 | Patches: 320 | Stride: 48

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9016 0.7287 0.7529 0.7815 0.8147 0.73  ]
Volume: 268x413x413 | Patches: 320 | Stride: 48

Case: case_20
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9185 0.8428 0.6502 0.72   0.763  0.8064]
Volume: 244x415x415 | Patches: 320 | Stride: 48

Case: case_17
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.8753 0.8036 0.7213 0.7663 0

### weights ([0.05, 1, 1.2, 1.5, 2, 2, 1])

In [14]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")

cbam nnU-Net style model Initialized


In [16]:

EPOCHS = 40

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)
weights = torch.tensor([0.05, 1, 1.2, 1.5, 2, 2, 1]).to(device)
# weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.6,
    class_weights = weights
)
scaler = torch.cuda.amp.GradScaler()


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_29364\2173923112.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [17]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "06_rw_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "06_rw_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "06_rw_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "06_rw_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "06_rw_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 198/198 [09:35<00:00,  2.91s/it]



Epoch 0
Train Loss: 0.9382
Val Loss: 0.2826
Per Class Dice: [0.53884514 0.51459565 0.55555556 0.33410541 0.37126998 0.58547054]
Mean Dice: 0.4722
Parotid Dice (L,R): 0.3341, 0.3713
Combined Score: 0.4363
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [08:12<00:00,  2.49s/it]



Epoch 1
Train Loss: 0.6188
Val Loss: 0.1586
Per Class Dice: [0.80287424 0.9062774  0.66916985 0.67393388 0.7815114  0.79816913]
Mean Dice: 0.7658
Parotid Dice (L,R): 0.6739, 0.7815
Combined Score: 0.7544
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [08:01<00:00,  2.43s/it]


...


Epoch 34
Train Loss: 0.2787
Val Loss: 0.0784
Per Class Dice: [0.82948811 0.86743032 0.86376828 0.86587107 0.93703614 0.89847533]
Mean Dice: 0.8865
Parotid Dice (L,R): 0.8659, 0.9370
Combined Score: 0.8910
Saved Best Dice Model
Saved Best Combined Model
LR: 0.000011


100%|██████████| 198/198 [08:40<00:00,  2.63s/it]



Epoch 35
Train Loss: 0.2569
Val Loss: 0.1543
Per Class Dice: [0.81818359 0.73628189 0.85134104 0.65973628 0.92181498 0.86718376]
Mean Dice: 0.8073
Parotid Dice (L,R): 0.6597, 0.9218
Combined Score: 0.8023
LR: 0.000008


 34%|███▍      | 67/198 [03:43<07:17,  3.34s/it]


KeyboardInterrupt: 

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM().to(device)

model_path = "../experiments/custom_loss_cbam/06_rw_best_combined_model"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_29364\2141417687.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)

Model loaded for testing.


In [23]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9303 0.8574 0.8114 0.8431 0.8453 0.9062]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9245 0.7753 0.73   0.8083 0.8228 0.8907]
Volume: 264x333x333 | Patches: 180 | Stride: 48

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9089 0.8375 0.8211 0.8432 0.8326 0.9049]
Volume: 248x395x395 | Patches: 320 | Stride: 48

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9156 0.7502 0.7504 0.8142 0.8147 0.7992]
Volume: 268x413x413 | Patches: 320 | Stride: 48

Case: case_20
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9262 0.8361 0.6537 0.7367 0.7446 0.8272]
Volume: 244x415x415 | Patches: 320 | Stride: 48

Case: case_17
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.881  0.81   0.7316 0.8086 0

### with spinal cord favor sampling 

In [9]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24  
).to(device)

print("cbam nnU-Net style model Initialized")

EPOCHS = 40

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)
weights = torch.tensor([0.05, 1, 1.3, 1.8, 2, 2, 1]).to(device)
# weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.6,
    class_weights = weights
)
scaler = torch.cuda.amp.GradScaler()


cbam nnU-Net style model Initialized


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_2520\3698800590.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [10]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "06_sc_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "06_sc_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "06_sc_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "06_sc_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "06_sc_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 198/198 [08:04<00:00,  2.44s/it]



Epoch 0
Train Loss: 0.9282
Val Loss: 0.2924
Per Class Dice: [0.56084444 0.60269137 0.51244803 0.35154642 0.44507598 0.61241748]
Mean Dice: 0.5048
Parotid Dice (L,R): 0.3515, 0.4451
Combined Score: 0.4729
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [08:32<00:00,  2.59s/it]



Epoch 1
Train Loss: 0.4953
Val Loss: 0.1000
Per Class Dice: [0.76858874 0.66871919 0.63581348 0.42556765 0.52042255 0.65682809]
Mean Dice: 0.5815
Parotid Dice (L,R): 0.4256, 0.5204
Combined Score: 0.5489
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [08:42<00:00,  2.64s/it]


...


Epoch 21
Train Loss: 0.2679
Val Loss: 0.0971
Per Class Dice: [0.80704388 0.81591713 0.81392843 0.93652967 0.87673854 0.87582591]
Mean Dice: 0.8638
Parotid Dice (L,R): 0.9365, 0.8767
Combined Score: 0.8766
Saved Best Parotid Model
LR: 0.000092


100%|██████████| 198/198 [13:55<00:00,  4.22s/it]



Epoch 22
Train Loss: 0.2710
Val Loss: 0.1105
Per Class Dice: [0.91638396 0.69968934 0.72147928 0.87054657 0.70880689 0.84798632]
Mean Dice: 0.7697
Parotid Dice (L,R): 0.8705, 0.7088
Combined Score: 0.7757
LR: 0.000084


 92%|█████████▏| 182/198 [13:00<01:08,  4.29s/it]


KeyboardInterrupt: 

In [12]:
def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location=device)

    # 🔥 Your format
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

        if optimizer is not None and "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        epoch = checkpoint.get("epoch", 0)

        # 🔥 support both names
        metric = checkpoint.get("metric", None)
        if metric is None:
            metric = checkpoint.get("best_val_loss", None)

    # 🔴 fallback (old case)
    else:
        model.load_state_dict(checkpoint)
        epoch = 0
        metric = None

    return epoch, metric

In [15]:
# Default
best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0
start_epoch = 0

import os
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

loss_path = os.path.join(SAVE_DIR, "best_loss_model.pth")
dice_path = os.path.join(SAVE_DIR, "best_dice_model.pth")
parotid_path = os.path.join(SAVE_DIR, "best_parotid_model.pth")
combined_path = os.path.join(SAVE_DIR, "06_sc_best_combined_model.pth")
last_path = os.path.join(SAVE_DIR, "last_model.pth")

if os.path.exists(loss_path):
    _, best_val_loss = load_checkpoint(loss_path, model)

if os.path.exists(dice_path):
    _, best_dice = load_checkpoint(dice_path, model)

if os.path.exists(parotid_path):
    _, best_parotid = load_checkpoint(parotid_path, model)

if os.path.exists(combined_path):
    _, best_combined = load_checkpoint(combined_path, model)



C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_32388\340703864.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


In [19]:

EPOCHS = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)
weights = torch.tensor([0.05, 1, 1.2, 1.5, 2, 2, 1]).to(device)
# weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.6,
    class_weights = weights
)
scaler = torch.cuda.amp.GradScaler()


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_32388\96275337.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [20]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "20custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "06_rw_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "06_rw_best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "06_rw_best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "06_rw_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "06_rw_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

NameError: name 'train_one_epoch_cbam' is not defined

In [16]:
print(best_dice) 
print(best_parotid)
print(best_combined)

0.8918156849013436
0.9066341055764092
0.8926435330179003


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM().to(device)

model_path = "../experiments/custom_loss_cbam/last_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_32388\1383292203.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)

nnUNet3D_CBAM(
  (enc1): ResidualBlock(
    (conv1): Conv3d(1, 24, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm1): InstanceNorm3d(24, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (act1): LeakyReLU(negative_slope=0.01, inplace=True)
    (conv2): Conv3d(24, 24, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm2): InstanceNorm3d(24, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (skip): Conv3d(1, 24, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (act2): LeakyReLU(negative_slope=0.01, inplace=True)
  )
  (enc2): ResidualBlock(
    (conv1): Conv3d(24, 48, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm1): InstanceNorm3d(48, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (act1): LeakyReLU(negative_slope=0.01, inplace=True)
    (conv2): Conv3d(48, 48, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm2): InstanceNorm3d(48, eps=1e-05, momen

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM().to(device)

model_path = "../experiments/custom_loss_cbam/06_sc_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_32388\2959247629.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)

Model loaded for testing.


In [11]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9285 0.8242 0.8013 0.8284 0.8412 0.8526]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9269 0.7061 0.7372 0.8227 0.8053 0.8735]
Volume: 264x333x333 | Patches: 180 | Stride: 48

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9089 0.8378 0.72   0.7023 0.8129 0.8877]
Volume: 248x395x395 | Patches: 320 | Stride: 48

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.908  0.6939 0.7496 0.8103 0.8144 0.7205]
Volume: 268x413x413 | Patches: 320 | Stride: 48

Case: case_20
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.92   0.7613 0.6664 0.726  0.7943 0.751 ]
Volume: 244x415x415 | Patches: 320 | Stride: 48

Case: case_17
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.8846 0.7076 0.7681 0.8342 0

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM().to(device)

model_path = "../experiments/custom_loss_cbam/06_iw_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\Yeseswini\AppData\Local\Temp\ipykernel_32388\911947872.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [24]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9368 0.848  0.8317 0.8594 0.8456 0.9266]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9287 0.7849 0.6974 0.8052 0.8283 0.8832]
Volume: 264x333x333 | Patches: 180 | Stride: 48

Case: case_22
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9129 0.8098 0.8317 0.8168 0.8261 0.9099]
Volume: 248x395x395 | Patches: 320 | Stride: 48

Case: case_14
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9116 0.6323 0.7689 0.8458 0.8323 0.8147]
Volume: 268x413x413 | Patches: 320 | Stride: 48

Case: case_20
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9229 0.8058 0.7544 0.7577 0.7571 0.8235]
Volume: 244x415x415 | Patches: 320 | Stride: 48

Case: case_17
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9027 0.6995 0.7262 0.8412 0